# Clinical Study RAG with Snowflake AI Observability

This notebook compares two simple versions of the same RECOVERY clinical-study RAG:

- **baseline_v1** — basic prompt, top_k = 3
- **grounded_v2** — strict grounding and abstention, top_k = 4

The goal is to demonstrate tracing, evaluation runs, RAG-triad metrics and version comparison.

In [ ]:
!pip install -r requirements.txt

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE WAREHOUSE CLINICAL_STUDY_AI_WH").collect()
session.sql("USE DATABASE CLINICAL_STUDY_AI_DB").collect()
session.sql("USE SCHEMA OBSERVABILITY").collect()
print("Session ready.")

In [ ]:
from typing import List, Dict
from snowflake.core import Root
from snowflake.cortex import Complete

from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes
from trulens.apps.app import TruApp
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core.run import Run, RunConfig

DB = "CLINICAL_STUDY_AI_DB"
RAG_SCHEMA = "RAG"
SEARCH_SERVICE = "CLINICAL_STUDY_SEARCH_SERVICE"
MODEL = "mistral-large2"

root = Root(session)

In [ ]:
class CortexSearchRetriever:
    def __init__(self, limit_to_retrieve: int):
        self.limit_to_retrieve = limit_to_retrieve

    def retrieve(self, query: str) -> List[Dict]:
        service = (
            root.databases[DB]
            .schemas[RAG_SCHEMA]
            .cortex_search_services[SEARCH_SERVICE]
        )
        response = service.search(
            query=query,
            columns=["CHUNK", "FILE_NAME", "DOCUMENT_TYPE", "CHUNK_INDEX"],
            limit=self.limit_to_retrieve
        )
        return response.results or []

In [ ]:
class ClinicalStudyRAG:
    def __init__(self, top_k: int, strict_grounding: bool):
        self.retriever = CortexSearchRetriever(top_k)
        self.strict_grounding = strict_grounding

    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        }
    )
    def retrieve_context(self, query: str) -> list:
        return self.retriever.retrieve(query)

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate_completion(self, query: str, retrieved_results: list) -> str:
        context = "\n\n".join(
            f"[Source: {item.get('FILE_NAME', 'Unknown')}]\n{item.get('CHUNK', '')}"
            for item in retrieved_results
        )

        if self.strict_grounding:
            instruction = '''
Use only the supplied RECOVERY trial context.
Do not use external knowledge.
Do not provide medical advice.
If the context does not support the answer, respond exactly:
"I could not find this information in the selected study documents."
Give a concise answer and mention the relevant source document names.
'''
        else:
            instruction = '''
Answer the question using the supplied clinical study context.
Give a clear and complete response.
'''

        prompt = f'''
You are a clinical study information assistant.

{instruction}

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:
'''
        return Complete(MODEL, prompt, session=session)

    @instrument(
        span_type=SpanAttributes.SpanType.RECORD_ROOT,
        attributes={
            SpanAttributes.RECORD_ROOT.INPUT: "query",
            SpanAttributes.RECORD_ROOT.OUTPUT: "return",
        }
    )
    def query(self, query: str) -> str:
        context = self.retrieve_context(query)
        return self.generate_completion(query, context)

## Test the two versions

In [ ]:
baseline_rag = ClinicalStudyRAG(top_k=3, strict_grounding=False)
grounded_rag = ClinicalStudyRAG(top_k=4, strict_grounding=True)

question = "What four general eligibility conditions must a patient meet?"

print("BASELINE\n", baseline_rag.query(question))
print("\nIMPROVED\n", grounded_rag.query(question))

## Register both versions

In [ ]:
connector = SnowflakeConnector(snowpark_session=session)
APP_NAME = "clinical_study_information_rag"

tru_baseline = TruApp(
    baseline_rag,
    app_name=APP_NAME,
    app_version="baseline_v1",
    connector=connector
)

tru_grounded = TruApp(
    grounded_rag,
    app_name=APP_NAME,
    app_version="grounded_v2",
    connector=connector
)

print("Application versions registered.")

## Create evaluation runs

In [ ]:
baseline_config = RunConfig(
    run_name="clinical_study_baseline_run",
    dataset_name="CLINICAL_STUDY_EVAL_DATA",
    description="Basic prompt with top_k=3",
    label="clinical_study_baseline",
    source_type="TABLE",
    dataset_spec={
        "input": "QUERY",
        "ground_truth_output": "GROUND_TRUTH_RESPONSE",
    },
)

grounded_config = RunConfig(
    run_name="clinical_study_grounded_run",
    dataset_name="CLINICAL_STUDY_EVAL_DATA",
    description="Strict grounding and abstention with top_k=4",
    label="clinical_study_grounded",
    source_type="TABLE",
    dataset_spec={
        "input": "QUERY",
        "ground_truth_output": "GROUND_TRUTH_RESPONSE",
    },
)

baseline_run: Run = tru_baseline.add_run(run_config=baseline_config)
grounded_run: Run = tru_grounded.add_run(run_config=grounded_config)

## Start both runs

In [ ]:
baseline_run.start()

In [ ]:
grounded_run.start()

## Compute RAG-triad metrics

In [ ]:
metrics = ["answer_relevance", "context_relevance", "groundedness"]

baseline_run.compute_metrics(metrics)
grounded_run.compute_metrics(metrics)

print("Metrics submitted.")

## Review results

Navigate to:

**AI & ML → Evaluations → clinical_study_information_rag**

Compare:

- baseline_v1 and grounded_v2
- average metric scores
- failed records
- retrieval spans
- generation spans
- retrieved contexts
- LLM-judge explanations